In [7]:
import pandas as pd

file_path = "../data/raw/transactions.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (852584, 14)

Columns:
['created_at', 'customer_id', 'booking_id', 'session_id', 'product_metadata', 'payment_method', 'payment_status', 'promo_amount', 'promo_code', 'shipment_fee', 'shipment_date_limit', 'shipment_location_lat', 'shipment_location_long', 'total_amount']


In [8]:
print("Payment status values:")
print(df["payment_status"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Payment status values:
payment_status
Success    815964
Failed      36620
Name: count, dtype: int64

Missing values:
created_at                     0
customer_id                    0
booking_id                     0
session_id                     0
product_metadata               0
payment_method                 0
payment_status                 0
promo_amount                   0
promo_code                526048
shipment_fee                   0
shipment_date_limit            0
shipment_location_lat          0
shipment_location_long         0
total_amount                   0
dtype: int64


In [9]:
print("Total transactions:", len(df))

print("Total transaction value:", df["total_amount"].sum())

print("Average transaction value:", df["total_amount"].mean())

print("Minimum transaction value:", df["total_amount"].min())

print("Maximum transaction value:", df["total_amount"].max())

Total transactions: 852584
Total transaction value: 468850022169
Average transaction value: 549916.5151691798
Minimum transaction value: 10898
Maximum transaction value: 23504487


In [10]:
successful = df[df["payment_status"] == "Success"]
failed = df[df["payment_status"] == "Failed"]

print("Successful transactions:", len(successful))
print("Failed transactions:", len(failed))

Successful transactions: 815964
Failed transactions: 36620


In [11]:
failed_revenue = failed["total_amount"].sum()

successful_revenue = successful["total_amount"].sum()

print("Successful revenue:", successful_revenue)
print("Failed revenue:", failed_revenue)

Successful revenue: 448997244108
Failed revenue: 19852778061


In [12]:
print(
    failed["payment_method"]
    .value_counts()
)

payment_method
Credit Card    12894
Gopay           7387
OVO             7058
Debit Card      5958
LinkAja         3323
Name: count, dtype: int64


In [13]:
print(
    failed.groupby("payment_method")["total_amount"]
    .agg(["count", "sum", "mean"])
    .sort_values("sum", ascending=False)
)

                count         sum           mean
payment_method                                  
Credit Card     12894  6928551931  537346.977742
Gopay            7387  3999892416  541477.245973
OVO              7058  3783849883  536107.946019
Debit Card       5958  3336039106  559925.999664
LinkAja          3323  1804444725  543016.769485


In [14]:
customer_transactions = (
    df.groupby("customer_id")
    .size()
    .reset_index(name="total_transactions")
)

print(customer_transactions.head())

   customer_id  total_transactions
0            3                  51
1            8                   7
2            9                   6
3           11                   1
4           15                   5


In [15]:
customer_success = (
    df.groupby("customer_id")["payment_status"]
    .apply(lambda x: (x == "Success").mean())
    .reset_index(name="success_rate")
)

print(customer_success.head())

   customer_id  success_rate
0            3      0.960784
1            8      1.000000
2            9      1.000000
3           11      1.000000
4           15      1.000000


In [16]:
customer_failures = (
    df[df["payment_status"] == "Failed"]
    .groupby("customer_id")
    .size()
    .reset_index(name="failed_transactions")
)

print(customer_failures.head())

   customer_id  failed_transactions
0            3                    2
1           18                    7
2           27                    2
3           67                    1
4           68                    6


In [17]:
customer_spending = (
    df.groupby("customer_id")["total_amount"]
    .agg(
        total_spent="sum",
        average_transaction="mean",
        transaction_count="count"
    )
    .reset_index()
)

print(customer_spending.head())


   customer_id  total_spent  average_transaction  transaction_count
0            3     21265889        416978.215686                 51
1            8      3898561        556937.285714                  7
2            9      2638665        439777.500000                  6
3           11       197533        197533.000000                  1
4           15      2134870        426974.000000                  5


In [18]:
customer_features = customer_transactions.merge(
    customer_success,
    on="customer_id",
    how="left"
)

customer_features = customer_features.merge(
    customer_failures,
    on="customer_id",
    how="left"
)

customer_features = customer_features.merge(
    customer_spending,
    on="customer_id",
    how="left"
)

customer_features["failed_transactions"] = (
    customer_features["failed_transactions"].fillna(0)
)

print(customer_features.head())
print("\nShape:", customer_features.shape)

   customer_id  total_transactions  success_rate  failed_transactions  \
0            3                  51      0.960784                  2.0   
1            8                   7      1.000000                  0.0   
2            9                   6      1.000000                  0.0   
3           11                   1      1.000000                  0.0   
4           15                   5      1.000000                  0.0   

   total_spent  average_transaction  transaction_count  
0     21265889        416978.215686                 51  
1      3898561        556937.285714                  7  
2      2638665        439777.500000                  6  
3       197533        197533.000000                  1  
4      2134870        426974.000000                  5  

Shape: (50705, 7)


In [19]:
df["created_at"] = pd.to_datetime(df["created_at"])

print(df["created_at"].dtype)
print(df["created_at"].min())
print(df["created_at"].max())

datetime64[us, UTC]
2016-06-30 23:18:44.792905+00:00
2022-07-31 23:59:45.821469+00:00


In [20]:
reference_date = df["created_at"].max()

print("Reference date:", reference_date)

Reference date: 2022-07-31 23:59:45.821469+00:00


In [21]:
last_transaction = (
    df.groupby("customer_id")["created_at"]
    .max()
    .reset_index(name="last_transaction")
)

last_transaction["recency_days"] = (
    reference_date - last_transaction["last_transaction"]
).dt.days

print(last_transaction.head())

   customer_id                 last_transaction  recency_days
0            3 2022-06-26 15:41:52.844494+00:00            35
1            8 2022-05-15 22:46:22.656991+00:00            77
2            9 2022-05-28 08:44:32.421034+00:00            64
3           11 2022-03-07 14:29:13.759159+00:00           146
4           15 2022-01-05 06:07:58.105040+00:00           207


In [22]:
customer_features = customer_features.merge(
    last_transaction[["customer_id", "recency_days"]],
    on="customer_id",
    how="left"
)

print(customer_features.head())

   customer_id  total_transactions  success_rate  failed_transactions  \
0            3                  51      0.960784                  2.0   
1            8                   7      1.000000                  0.0   
2            9                   6      1.000000                  0.0   
3           11                   1      1.000000                  0.0   
4           15                   5      1.000000                  0.0   

   total_spent  average_transaction  transaction_count  recency_days  
0     21265889        416978.215686                 51            35  
1      3898561        556937.285714                  7            77  
2      2638665        439777.500000                  6            64  
3       197533        197533.000000                  1           146  
4      2134870        426974.000000                  5           207  


In [23]:
print("Customer feature shape:", customer_features.shape)

print("\nCustomer features:")
print(customer_features.columns.tolist())

print("\nMissing values:")
print(customer_features.isnull().sum())

Customer feature shape: (50705, 8)

Customer features:
['customer_id', 'total_transactions', 'success_rate', 'failed_transactions', 'total_spent', 'average_transaction', 'transaction_count', 'recency_days']

Missing values:
customer_id            0
total_transactions     0
success_rate           0
failed_transactions    0
total_spent            0
average_transaction    0
transaction_count      0
recency_days           0
dtype: int64


In [24]:
ml_data = df.merge(
    customer_features,
    on="customer_id",
    how="left"
)

print("ML dataset shape:", ml_data.shape)

ML dataset shape: (852584, 21)


In [25]:
ml_data["payment_failed"] = (
    ml_data["payment_status"] == "Failed"
).astype(int)

print(ml_data["payment_failed"].value_counts())

payment_failed
0    815964
1     36620
Name: count, dtype: int64


In [26]:
import numpy as np
ml_data["amount_log"] = np.log1p(ml_data["total_amount"])

ml_data["amount_vs_customer_avg"] = (
    ml_data["total_amount"] /
    ml_data["average_transaction"].replace(0, np.nan)
)

ml_data["amount_vs_customer_avg"] = (
    ml_data["amount_vs_customer_avg"].fillna(1)
)

In [27]:
ml_data["failure_rate"] = (
    ml_data["failed_transactions"] /
    ml_data["total_transactions"]
)

ml_data["failure_rate"] = (
    ml_data["failure_rate"].fillna(0)
)

In [28]:
print("Shape:", ml_data.shape)

print("\nML columns:")
print(ml_data.columns.tolist())

print("\nTarget distribution:")
print(ml_data["payment_failed"].value_counts())

print("\nSample:")
print(
    ml_data[
        [
            "customer_id",
            "total_amount",
            "payment_method",
            "payment_status",
            "success_rate",
            "failed_transactions",
            "recency_days",
            "failure_rate",
            "payment_failed"
        ]
    ].head()
)

Shape: (852584, 25)

ML columns:
['created_at', 'customer_id', 'booking_id', 'session_id', 'product_metadata', 'payment_method', 'payment_status', 'promo_amount', 'promo_code', 'shipment_fee', 'shipment_date_limit', 'shipment_location_lat', 'shipment_location_long', 'total_amount', 'total_transactions', 'success_rate', 'failed_transactions', 'total_spent', 'average_transaction', 'transaction_count', 'recency_days', 'payment_failed', 'amount_log', 'amount_vs_customer_avg', 'failure_rate']

Target distribution:
payment_failed
0    815964
1     36620
Name: count, dtype: int64

Sample:
   customer_id  total_amount payment_method payment_status  success_rate  \
0         5868        199832     Debit Card        Success           1.0   
1         4774        155526    Credit Card        Success           1.0   
2         4774        550696            OVO        Success           1.0   
3         4774        271012    Credit Card        Success           1.0   
4         4774        198753   

In [29]:
print(ml_data.dtypes)

created_at                datetime64[us, UTC]
customer_id                             int64
booking_id                                str
session_id                                str
product_metadata                          str
payment_method                            str
payment_status                            str
promo_amount                            int64
promo_code                                str
shipment_fee                            int64
shipment_date_limit                       str
shipment_location_lat                 float64
shipment_location_long                float64
total_amount                            int64
total_transactions                      int64
success_rate                          float64
failed_transactions                   float64
total_spent                             int64
average_transaction                   float64
transaction_count                       int64
recency_days                            int64
payment_failed                    

In [30]:
missing = ml_data.isnull().sum()

print(
    missing[missing > 0].sort_values(ascending=False)
)

promo_code    526048
dtype: int64


In [31]:
ml_data["promo_code"] = (
    ml_data["promo_code"].fillna("NO_PROMO")
)
print(ml_data["promo_code"].isnull().sum())

0


In [32]:
model_data = ml_data[
    [
        "total_amount",
        "amount_log",
        "amount_vs_customer_avg",
        "success_rate",
        "failed_transactions",
        "recency_days",
        "failure_rate",
        "payment_method",
        "promo_amount",
        "shipment_fee",
        "payment_failed"
    ]
].copy()

print(model_data.head())
print("\nShape:", model_data.shape)

   total_amount  amount_log  amount_vs_customer_avg  success_rate  \
0        199832   12.205237                1.000000           1.0   
1        155526   11.954575                0.320593           1.0   
2        550696   13.218940                1.135176           1.0   
3        271012   12.509922                0.558650           1.0   
4        198753   12.199823                0.409699           1.0   

   failed_transactions  recency_days  failure_rate payment_method  \
0                  0.0          1463           0.0     Debit Card   
1                  0.0            52           0.0    Credit Card   
2                  0.0            52           0.0            OVO   
3                  0.0            52           0.0    Credit Card   
4                  0.0            52           0.0    Credit Card   

   promo_amount  shipment_fee  payment_failed  
0          1415         10000               0  
1             0         10000               0  
2             0         10

In [33]:
print("Missing values in model data:")

print(
    model_data.isnull().sum()
)

Missing values in model data:
total_amount              0
amount_log                0
amount_vs_customer_avg    0
success_rate              0
failed_transactions       0
recency_days              0
failure_rate              0
payment_method            0
promo_amount              0
shipment_fee              0
payment_failed            0
dtype: int64


In [34]:
print(model_data["payment_method"].value_counts())

payment_method
Credit Card    299586
Gopay          171334
OVO            169066
Debit Card     137269
LinkAja         75329
Name: count, dtype: int64


In [35]:
print("Number of payment methods:", model_data["payment_method"].nunique())

Number of payment methods: 5


In [36]:
model_data = pd.get_dummies(
    model_data,
    columns=["payment_method"],
    drop_first=True
)

print(model_data.head())

   total_amount  amount_log  amount_vs_customer_avg  success_rate  \
0        199832   12.205237                1.000000           1.0   
1        155526   11.954575                0.320593           1.0   
2        550696   13.218940                1.135176           1.0   
3        271012   12.509922                0.558650           1.0   
4        198753   12.199823                0.409699           1.0   

   failed_transactions  recency_days  failure_rate  promo_amount  \
0                  0.0          1463           0.0          1415   
1                  0.0            52           0.0             0   
2                  0.0            52           0.0             0   
3                  0.0            52           0.0             0   
4                  0.0            52           0.0             0   

   shipment_fee  payment_failed  payment_method_Debit Card  \
0         10000               0                       True   
1         10000               0                     

In [37]:
bool_columns = model_data.select_dtypes(
    include="bool"
).columns

model_data[bool_columns] = (
    model_data[bool_columns].astype(int)
)

In [38]:
print("Final columns:")
print(model_data.columns.tolist())

print("\nData types:")
print(model_data.dtypes)

print("\nShape:")
print(model_data.shape)

Final columns:
['total_amount', 'amount_log', 'amount_vs_customer_avg', 'success_rate', 'failed_transactions', 'recency_days', 'failure_rate', 'promo_amount', 'shipment_fee', 'payment_failed', 'payment_method_Debit Card', 'payment_method_Gopay', 'payment_method_LinkAja', 'payment_method_OVO']

Data types:
total_amount                   int64
amount_log                   float64
amount_vs_customer_avg       float64
success_rate                 float64
failed_transactions          float64
recency_days                   int64
failure_rate                 float64
promo_amount                   int64
shipment_fee                   int64
payment_failed                 int64
payment_method_Debit Card      int64
payment_method_Gopay           int64
payment_method_LinkAja         int64
payment_method_OVO             int64
dtype: object

Shape:
(852584, 14)


In [39]:
df = df.sort_values("created_at").reset_index(drop=True)

print(df[["created_at", "customer_id", "payment_status"]].head())

                        created_at  customer_id payment_status
0 2016-06-30 23:18:44.792905+00:00        74089        Success
1 2016-07-01 02:53:26.720195+00:00         6183        Success
2 2016-07-01 09:45:51.803842+00:00        16228        Success
3 2016-07-01 22:32:45.218400+00:00        73773        Success
4 2016-07-02 05:28:40.302335+00:00        37770        Success


In [40]:
df["payment_failed"] = (
    df["payment_status"] == "Failed"
).astype(int)

df["previous_transactions"] = (
    df.groupby("customer_id")
      .cumcount()
)

df["previous_failures"] = (
    df.groupby("customer_id")["payment_failed"]
      .cumsum()
      .shift(1)
      .fillna(0)
)

df["previous_successes"] = (
    df["previous_transactions"] -
    df["previous_failures"]
)

In [41]:
df["previous_failure_rate"] = (
    df["previous_failures"] /
    df["previous_transactions"].replace(0, np.nan)
)

df["previous_failure_rate"] = (
    df["previous_failure_rate"].fillna(0)
)

print(
    df[
        [
            "customer_id",
            "payment_status",
            "previous_transactions",
            "previous_failures",
            "previous_failure_rate"
        ]
    ].head(20)
)

    customer_id payment_status  previous_transactions  previous_failures  \
0         74089        Success                      0                0.0   
1          6183        Success                      0                0.0   
2         16228        Success                      0                0.0   
3         73773        Success                      0                0.0   
4         37770        Success                      0                0.0   
5          2181        Success                      0                0.0   
6           933        Success                      0                0.0   
7         30771        Success                      0                0.0   
8         68632        Success                      0                0.0   
9         81830        Success                      0                0.0   
10        95233        Success                      0                0.0   
11        88717        Success                      0                0.0   
12        67

In [42]:
df["previous_spending"] = (
    df.groupby("customer_id")["total_amount"]
      .cumsum()
      .shift(1)
      .fillna(0)
)

df["previous_average_amount"] = (
    df["previous_spending"] /
    df["previous_transactions"].replace(0, np.nan)
)

df["previous_average_amount"] = (
    df["previous_average_amount"].fillna(0)
)

print(
    df[
        [
            "customer_id",
            "total_amount",
            "previous_spending",
            "previous_average_amount"
        ]
    ].head(20)
)

    customer_id  total_amount  previous_spending  previous_average_amount
0         74089        640795                0.0                      0.0
1          6183        578826           640795.0                      0.0
2         16228        333792           578826.0                      0.0
3         73773        256670           333792.0                      0.0
4         37770        543010           256670.0                      0.0
5          2181        304345           543010.0                      0.0
6           933        237978           304345.0                      0.0
7         30771        199897           237978.0                      0.0
8         68632        282734           199897.0                      0.0
9         81830         68935           282734.0                      0.0
10        95233        116206            68935.0                      0.0
11        88717        206360           116206.0                      0.0
12        67563        996610         

In [43]:
ml_data = df[
    [
        "total_amount",
        "payment_method",
        "promo_amount",
        "shipment_fee",
        "previous_transactions",
        "previous_failures",
        "previous_successes",
        "previous_failure_rate",
        "previous_spending",
        "previous_average_amount",
        "payment_failed"
    ]
].copy()

print("Corrected ML dataset shape:", ml_data.shape)

print("\nMissing values:")
print(ml_data.isnull().sum())

Corrected ML dataset shape: (852584, 11)

Missing values:
total_amount               0
payment_method             0
promo_amount               0
shipment_fee               0
previous_transactions      0
previous_failures          0
previous_successes         0
previous_failure_rate      0
previous_spending          0
previous_average_amount    0
payment_failed             0
dtype: int64


In [44]:
X = ml_data.drop("payment_failed", axis=1)
y = ml_data["payment_failed"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (852584, 10)
Target shape: (852584,)


In [45]:
print(y.value_counts())
print("\nTarget percentages:")
print(y.value_counts(normalize=True) * 100)

payment_failed
0    815964
1     36620
Name: count, dtype: int64

Target percentages:
payment_failed
0    95.704822
1     4.295178
Name: proportion, dtype: float64


In [46]:
print(y.value_counts())
print("\nTarget percentages:")
print(y.value_counts(normalize=True) * 100)

payment_failed
0    815964
1     36620
Name: count, dtype: int64

Target percentages:
payment_failed
0    95.704822
1     4.295178
Name: proportion, dtype: float64


In [47]:
X = pd.get_dummies(
    X,
    columns=["payment_method"],
    drop_first=True
)

In [48]:
bool_columns = X.select_dtypes(include="bool").columns

X[bool_columns] = X[bool_columns].astype(int)

In [49]:
from sklearn.model_selection import train_test_split

In [50]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 682067
Testing rows: 170517


In [51]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training target distribution:
payment_failed
0    95.704821
1     4.295179
Name: proportion, dtype: float64

Testing target distribution:
payment_failed
0    95.704827
1     4.295173
Name: proportion, dtype: float64


In [52]:
df["previous_failures"] = (
    df.groupby("customer_id")["payment_failed"]
      .transform(lambda x: x.shift(1).fillna(0).cumsum())
)

In [53]:
df["previous_transactions"] = (
    df.groupby("customer_id")
      .cumcount()
)

In [54]:
df["previous_successes"] = (
    df["previous_transactions"] -
    df["previous_failures"]
)

In [55]:
df["previous_spending"] = (
    df.groupby("customer_id")["total_amount"]
      .transform(lambda x: x.shift(1).fillna(0).cumsum())
)

In [56]:
df["previous_average_amount"] = (
    df["previous_spending"] /
    df["previous_transactions"].replace(0, np.nan)
)

df["previous_average_amount"] = (
    df["previous_average_amount"].fillna(0)
)

In [57]:
df["previous_average_amount"] = (
    df["previous_spending"] /
    df["previous_transactions"].replace(0, np.nan)
)

df["previous_average_amount"] = (
    df["previous_average_amount"].fillna(0)
)

In [58]:
print(
    df[
        [
            "customer_id",
            "total_amount",
            "previous_transactions",
            "previous_failures",
            "previous_successes",
            "previous_spending",
            "previous_average_amount"
        ]
    ].head(20)
)

    customer_id  total_amount  previous_transactions  previous_failures  \
0         74089        640795                      0                0.0   
1          6183        578826                      0                0.0   
2         16228        333792                      0                0.0   
3         73773        256670                      0                0.0   
4         37770        543010                      0                0.0   
5          2181        304345                      0                0.0   
6           933        237978                      0                0.0   
7         30771        199897                      0                0.0   
8         68632        282734                      0                0.0   
9         81830         68935                      0                0.0   
10        95233        116206                      0                0.0   
11        88717        206360                      0                0.0   
12        67563        99

In [59]:
customer_counts = df["customer_id"].value_counts()

print(customer_counts.head(10))

customer_id
43202    550
29496    505
82237    503
10167    473
69740    458
38588    436
64659    426
29240    402
95492    401
20143    399
Name: count, dtype: int64


In [60]:
repeat_customer = customer_counts[customer_counts > 3].index[0]

print("Test customer:", repeat_customer)

Test customer: 43202


In [61]:
print(
    df[df["customer_id"] == repeat_customer][
        [
            "created_at",
            "customer_id",
            "total_amount",
            "payment_status",
            "previous_transactions",
            "previous_failures",
            "previous_successes",
            "previous_spending",
            "previous_average_amount"
        ]
    ].head(15).to_string(index=False)
)


                      created_at  customer_id  total_amount payment_status  previous_transactions  previous_failures  previous_successes  previous_spending  previous_average_amount
2016-07-24 22:42:00.459435+00:00        43202        354853        Success                      0                0.0                 0.0                0.0             0.000000e+00
2016-07-28 22:29:25.459435+00:00        43202        400824        Success                      1                0.0                 1.0           354853.0             3.548530e+05
2016-08-01 22:31:04.459435+00:00        43202        142226        Success                      2                0.0                 2.0           755677.0             3.778385e+05
2016-08-05 22:37:58.459435+00:00        43202       3574144        Success                      3                0.0                 3.0           897903.0             2.993010e+05
2016-08-09 22:32:29.459435+00:00        43202        116369        Success                     

In [62]:
print(
    df[df["customer_id"] == repeat_customer][
        [
            "payment_status",
            "previous_transactions",
            "previous_failures",
            "previous_failure_rate"
        ]
    ].head(20).to_string(index=False)
)

payment_status  previous_transactions  previous_failures  previous_failure_rate
       Success                      0                0.0                    0.0
       Success                      1                0.0                    1.0
       Success                      2                0.0                    0.0
       Success                      3                0.0                    0.0
       Success                      4                0.0                    0.0
       Success                      5                0.0                    0.0
       Success                      6                0.0                    0.0
       Success                      7                0.0                    0.0
       Success                      8                0.0                    0.0
       Success                      9                0.0                    0.0
       Success                     10                0.0                    0.0
       Success                     11   

In [63]:
ml_data = df[
    [
        "total_amount",
        "payment_method",
        "promo_amount",
        "shipment_fee",
        "previous_transactions",
        "previous_failures",
        "previous_successes",
        "previous_failure_rate",
        "previous_spending",
        "previous_average_amount",
        "payment_failed"
    ]
].copy()

print("ML data shape:", ml_data.shape)

print("\nMissing values:")
print(ml_data.isnull().sum())

ML data shape: (852584, 11)

Missing values:
total_amount               0
payment_method             0
promo_amount               0
shipment_fee               0
previous_transactions      0
previous_failures          0
previous_successes         0
previous_failure_rate      0
previous_spending          0
previous_average_amount    0
payment_failed             0
dtype: int64


In [64]:
ml_data = df[
    [
        "created_at",
        "total_amount",
        "payment_method",
        "promo_amount",
        "shipment_fee",
        "previous_transactions",
        "previous_failures",
        "previous_successes",
        "previous_failure_rate",
        "previous_spending",
        "previous_average_amount",
        "payment_failed"
    ]
].copy()

ml_data = ml_data.sort_values("created_at").reset_index(drop=True)

print(ml_data.shape)
print(ml_data["created_at"].head())

(852584, 12)
0   2016-06-30 23:18:44.792905+00:00
1   2016-07-01 02:53:26.720195+00:00
2   2016-07-01 09:45:51.803842+00:00
3   2016-07-01 22:32:45.218400+00:00
4   2016-07-02 05:28:40.302335+00:00
Name: created_at, dtype: datetime64[us, UTC]


In [65]:
ml_data = pd.get_dummies(
    ml_data,
    columns=["payment_method"],
    drop_first=True
)

bool_columns = ml_data.select_dtypes(include="bool").columns

ml_data[bool_columns] = (
    ml_data[bool_columns].astype(int)
)

print(ml_data.dtypes)

created_at                   datetime64[us, UTC]
total_amount                               int64
promo_amount                               int64
shipment_fee                               int64
previous_transactions                      int64
previous_failures                        float64
previous_successes                       float64
previous_failure_rate                    float64
previous_spending                        float64
previous_average_amount                  float64
payment_failed                             int64
payment_method_Debit Card                  int64
payment_method_Gopay                       int64
payment_method_LinkAja                     int64
payment_method_OVO                         int64
dtype: object


In [66]:
split_index = int(len(ml_data) * 0.80)

train_data = ml_data.iloc[:split_index].copy()
test_data = ml_data.iloc[split_index:].copy()

print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

print("\nTraining period:")
print(train_data["created_at"].min(), "to", train_data["created_at"].max())

print("\nTesting period:")
print(test_data["created_at"].min(), "to", test_data["created_at"].max())

Training rows: 682067
Testing rows: 170517

Training period:
2016-06-30 23:18:44.792905+00:00 to 2022-01-28 19:46:08.531446+00:00

Testing period:
2022-01-28 19:46:34.252631+00:00 to 2022-07-31 23:59:45.821469+00:00


In [67]:
X_train = train_data.drop(
    ["payment_failed", "created_at"],
    axis=1
)

y_train = train_data["payment_failed"]

X_test = test_data.drop(
    ["payment_failed", "created_at"],
    axis=1
)

y_test = test_data["payment_failed"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (682067, 13)
y_train: (682067,)
X_test: (170517, 13)
y_test: (170517,)


In [68]:
print("Training target:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True) * 100)

Training target:
payment_failed
0    652872
1     29195
Name: count, dtype: int64
payment_failed
0    95.719629
1     4.280371
Name: proportion, dtype: float64

Testing target:
payment_failed
0    163092
1      7425
Name: count, dtype: int64
payment_failed
0    95.645595
1     4.354405
Name: proportion, dtype: float64


In [69]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [70]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

In [71]:
print("Training baseline model...")

baseline_model.fit(X_train, y_train)

print("Baseline model trained successfully!")

Training baseline model...
Baseline model trained successfully!


In [72]:
y_pred = baseline_model.predict(X_test)

y_prob = baseline_model.predict_proba(X_test)[:, 1]

print("Predictions generated.")

Predictions generated.


In [73]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nROC-AUC:")
print(roc_auc_score(y_test, y_prob))

print("\nPR-AUC:")
print(average_precision_score(y_test, y_prob))

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.78      0.87    163092
           1       0.11      0.57      0.18      7425

    accuracy                           0.77    170517
   macro avg       0.54      0.67      0.52    170517
weighted avg       0.94      0.77      0.84    170517


Confusion Matrix:
[[127132  35960]
 [  3195   4230]]

ROC-AUC:
0.7358965921281669

PR-AUC:
0.11696002852453258


In [74]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[127132  35960]
 [  3195   4230]]


In [75]:
roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.7358965921281669


In [76]:
pr_auc = average_precision_score(y_test, y_prob)

print("PR-AUC:", pr_auc)

PR-AUC: 0.11696002852453258


In [77]:
print("Minimum probability:", y_prob.min())
print("Maximum probability:", y_prob.max())
print("Average probability:", y_prob.mean())

Minimum probability: 0.00020257378848302713
Maximum probability: 0.9999999752243831
Average probability: 0.4416834402227775


In [78]:
risk_check = test_data.copy()

risk_check["failure_probability"] = y_prob

risk_check["payment_method"] = df.loc[
    test_data.index,
    "payment_method"
].values

risk_check = risk_check.sort_values(
    "failure_probability",
    ascending=False
)

print(
    risk_check[
        [
            "created_at",
            "total_amount",
            "payment_method",
            "previous_transactions",
            "previous_failures",
            "previous_failure_rate",
            "failure_probability",
            "payment_failed"
        ]
    ].head(20).to_string(index=False)
)

                      created_at  total_amount payment_method  previous_transactions  previous_failures  previous_failure_rate  failure_probability  payment_failed
2022-06-15 03:26:37.244382+00:00       1146453    Credit Card                    276               83.0               0.000000                  1.0               0
2022-06-22 03:17:33.244382+00:00        300722    Credit Card                    277               83.0               0.000000                  1.0               0
2022-06-29 03:42:21.244382+00:00        600778    Credit Card                    278               83.0               0.025180                  1.0               0
2022-07-06 03:49:38.244382+00:00        993694    Credit Card                    279               83.0               0.000000                  1.0               0
2022-07-27 04:01:50.244382+00:00        160292    Credit Card                    282               83.0               0.010638                  1.0               0
2022-07-13 03:50

In [79]:
check = df[df["previous_transactions"] > 0].copy()

check["calculated_rate"] = (
    check["previous_failures"] /
    check["previous_transactions"]
)

print(
    check[
        [
            "customer_id",
            "previous_transactions",
            "previous_failures",
            "previous_failure_rate",
            "calculated_rate"
        ]
    ].head(20).to_string(index=False)
)

 customer_id  previous_transactions  previous_failures  previous_failure_rate  calculated_rate
       95233                      1                0.0                    0.0              0.0
       83397                      1                0.0                    0.0              0.0
       47098                      1                0.0                    0.0              0.0
       95233                      2                0.0                    0.0              0.0
       13741                      1                0.0                    0.0              0.0
       83397                      2                0.0                    0.5              0.0
       21958                      1                0.0                    0.0              0.0
       55524                      1                0.0                    0.0              0.0
        6183                      1                0.0                    0.0              0.0
       85670                      1               

In [80]:
failed_history = df[
    df["previous_failures"] > 0
]

print(
    failed_history[
        [
            "customer_id",
            "created_at",
            "payment_status",
            "previous_transactions",
            "previous_failures",
            "previous_failure_rate"
        ]
    ].head(20).to_string(index=False)
)

 customer_id                       created_at payment_status  previous_transactions  previous_failures  previous_failure_rate
       17556 2016-08-25 11:01:05.332885+00:00        Success                      1                1.0                    0.0
       86386 2016-08-27 18:23:25.908412+00:00        Success                      5                1.0                    0.0
       31173 2016-08-29 08:50:43.974857+00:00        Success                      1                1.0                    0.0
       47098 2016-08-29 11:54:34.368694+00:00        Success                      4                1.0                    0.0
       14484 2016-08-31 09:12:14.914971+00:00        Success                      1                1.0                    0.0
       35089 2016-08-31 10:24:30.886753+00:00        Success                      3                1.0                    0.0
       55709 2016-08-31 20:03:53.941766+00:00        Success                      1                1.0                

In [81]:
print(
    "Transactions with previous failures:",
    len(failed_history)
)

print(
    "Maximum previous failures:",
    df["previous_failures"].max()
)

print(
    "Maximum previous transactions:",
    df["previous_transactions"].max()
)

Transactions with previous failures: 317078
Maximum previous failures: 83.0
Maximum previous transactions: 549


In [82]:
df["previous_failure_rate"] = (
    df["previous_failures"] /
    df["previous_transactions"].replace(0, np.nan)
).fillna(0)

In [83]:
ml_data = df[
    [
        "created_at",
        "total_amount",
        "payment_method",
        "promo_amount",
        "shipment_fee",
        "previous_transactions",
        "previous_failures",
        "previous_successes",
        "previous_failure_rate",
        "previous_spending",
        "previous_average_amount",
        "payment_failed"
    ]
].copy()

ml_data = ml_data.sort_values(
    "created_at"
).reset_index(drop=True)

print("ML data shape:", ml_data.shape)

ML data shape: (852584, 12)


In [84]:
ml_data = pd.get_dummies(
    ml_data,
    columns=["payment_method"],
    drop_first=True
)

bool_columns = ml_data.select_dtypes(
    include="bool"
).columns

ml_data[bool_columns] = (
    ml_data[bool_columns].astype(int)
)

In [85]:
split_index = int(len(ml_data) * 0.80)

train_data = ml_data.iloc[:split_index].copy()
test_data = ml_data.iloc[split_index:].copy()

X_train = train_data.drop(
    ["payment_failed", "created_at"],
    axis=1
)

y_train = train_data["payment_failed"]

X_test = test_data.drop(
    ["payment_failed", "created_at"],
    axis=1
)

y_test = test_data["payment_failed"]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (682067, 13)
Testing: (170517, 13)


In [86]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

print("Training corrected baseline...")

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")

print("\nROC-AUC:")
print(roc_auc_score(y_test, y_prob))

print("\nPR-AUC:")
print(average_precision_score(y_test, y_prob))

print("\nProbability range:")
print("Min:", y_prob.min())
print("Max:", y_prob.max())
print("Mean:", y_prob.mean())

Training corrected baseline...
Model trained successfully.

ROC-AUC:
0.7404561710268919

PR-AUC:
0.12116944719047462

Probability range:
Min: 0.0022184133415047865
Max: 0.9999950974900895
Mean: 0.4379020843515594


In [87]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.79      0.87    163092
           1       0.11      0.56      0.18      7425

    accuracy                           0.78    170517
   macro avg       0.54      0.67      0.53    170517
weighted avg       0.94      0.78      0.84    170517



In [88]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[129022  34070]
 [  3287   4138]]


In [89]:
from sklearn.metrics import recall_score, precision_score, f1_score

failure_recall = recall_score(y_test, y_pred)
failure_precision = precision_score(y_test, y_pred)
failure_f1 = f1_score(y_test, y_pred)

print("Failure Recall:", failure_recall)
print("Failure Precision:", failure_precision)
print("Failure F1:", failure_f1)

Failure Recall: 0.5573063973063973
Failure Precision: 0.10830192629815745
Failure F1: 0.18135998071571013


In [90]:
evaluation_data = test_data.copy()

evaluation_data["predicted_failure"] = y_pred
evaluation_data["failure_probability"] = y_prob

true_failed = evaluation_data[
    evaluation_data["payment_failed"] == 1
]

correctly_identified = evaluation_data[
    (evaluation_data["payment_failed"] == 1) &
    (evaluation_data["predicted_failure"] == 1)
]

print("Actual failed transactions:", len(true_failed))

print(
    "Correctly identified failures:",
    len(correctly_identified)
)

print(
    "Total failed transaction value:",
    true_failed["total_amount"].sum()
)

print(
    "Correctly identified failed transaction value:",
    correctly_identified["total_amount"].sum()
)

Actual failed transactions: 7425
Correctly identified failures: 4138
Total failed transaction value: 4041413023
Correctly identified failed transaction value: 2228746213


In [91]:
revenue_capture_rate = (
    correctly_identified["total_amount"].sum()
    / true_failed["total_amount"].sum()
)

print(
    "Failed-revenue capture rate:",
    revenue_capture_rate
)

print(
    "Failed-revenue capture percentage:",
    revenue_capture_rate * 100,
    "%"
)

Failed-revenue capture rate: 0.5514769711276798
Failed-revenue capture percentage: 55.14769711276798 %


In [92]:
import numpy as np

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):

    predictions = (
        y_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(
    threshold_results
)

threshold_results

,threshold,precision,recall,f1
0,0.10,0.044046,0.998249,0.084370
1,0.15,0.044494,0.995421,0.085181
2,0.20,0.045283,0.991515,0.086611
3,0.25,0.046620,0.985455,0.089029
4,0.30,0.048704,0.971582,0.092758
5,0.35,0.052713,0.942357,0.099841
6,0.40,0.062550,0.868148,0.116693
7,0.45,0.091761,0.663165,0.161215
8,0.50,0.108302,0.557306,0.181360
9,0.55,0.120019,0.481077,0.192110


In [93]:
high_recall = threshold_results[
    threshold_results["recall"] >= 0.70
]

print(high_recall)

   threshold  precision    recall        f1
0       0.10   0.044046  0.998249  0.084370
1       0.15   0.044494  0.995421  0.085181
2       0.20   0.045283  0.991515  0.086611
3       0.25   0.046620  0.985455  0.089029
4       0.30   0.048704  0.971582  0.092758
5       0.35   0.052713  0.942357  0.099841
6       0.40   0.062550  0.868148  0.116693


In [94]:
useful_thresholds = threshold_results[
    threshold_results["recall"] >= 0.50
]

best_precision = useful_thresholds.loc[
    useful_thresholds["precision"].idxmax()
]

print(best_precision)

threshold    0.500000
precision    0.108302
recall       0.557306
f1           0.181360
Name: 8, dtype: float64


In [95]:
print(
    threshold_results.to_string(
        index=False
    )
)

 threshold  precision   recall       f1
      0.10   0.044046 0.998249 0.084370
      0.15   0.044494 0.995421 0.085181
      0.20   0.045283 0.991515 0.086611
      0.25   0.046620 0.985455 0.089029
      0.30   0.048704 0.971582 0.092758
      0.35   0.052713 0.942357 0.099841
      0.40   0.062550 0.868148 0.116693
      0.45   0.091761 0.663165 0.161215
      0.50   0.108302 0.557306 0.181360
      0.55   0.120019 0.481077 0.192110
      0.60   0.132158 0.408754 0.199737
      0.65   0.142972 0.334411 0.200307
      0.70   0.155103 0.269764 0.196962
      0.75   0.165442 0.212256 0.185948
      0.80   0.174371 0.165118 0.169618
      0.85   0.186268 0.115825 0.142833
      0.90   0.201054 0.077037 0.111392


In [96]:
recovery_data = test_data.copy()

recovery_data["failure_probability"] = y_prob

print(
    recovery_data[
        [
            "total_amount",
            "failure_probability",
            "payment_failed"
        ]
    ].head()
)

        total_amount  failure_probability  payment_failed
682067        145961             0.298083               0
682068       2568181             0.406599               0
682069        356326             0.603790               0
682070        252194             0.344768               0
682071        210526             0.218969               0


In [97]:
# Rebuild recovery_data from scratch

recovery_data = test_data.copy()

print("Step 1 - recovery_data created")
print(recovery_data.shape)

# Add predicted failure probability
recovery_data["failure_probability"] = y_prob

print("Step 2 - probability added")

# Add revenue at risk
recovery_data["revenue_at_risk"] = (
    recovery_data["total_amount"]
    * recovery_data["failure_probability"]
)

print("Step 3 - revenue_at_risk added")

# Check that the column exists
print("\nColumns:")
print(recovery_data.columns.tolist())

# Now sort
recovery_data = recovery_data.sort_values(
    "revenue_at_risk",
    ascending=False
).reset_index(drop=True)

print("\nTop 10 highest-risk transactions:")

print(
    recovery_data[
        [
            "total_amount",
            "failure_probability",
            "revenue_at_risk",
            "payment_failed"
        ]
    ].head(10).to_string(index=False)
)

Step 1 - recovery_data created
(170517, 15)
Step 2 - probability added
Step 3 - revenue_at_risk added

Columns:
['created_at', 'total_amount', 'promo_amount', 'shipment_fee', 'previous_transactions', 'previous_failures', 'previous_successes', 'previous_failure_rate', 'previous_spending', 'previous_average_amount', 'payment_failed', 'payment_method_Debit Card', 'payment_method_Gopay', 'payment_method_LinkAja', 'payment_method_OVO', 'failure_probability', 'revenue_at_risk']

Top 10 highest-risk transactions:
 total_amount  failure_probability  revenue_at_risk  payment_failed
     13249933             0.818853     1.084974e+07               0
     12301668             0.867448     1.067106e+07               0
     13828846             0.770681     1.065762e+07               1
     14587394             0.679455     9.911477e+06               0
     14876129             0.648219     9.642992e+06               0
     15270856             0.613130     9.363020e+06               0
     1049992

In [98]:
print("Kernel is working")

Kernel is working


In [99]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print("Libraries loaded")

Libraries loaded


In [100]:
print("df exists:", "df" in globals())

df exists: True


In [101]:
import os

print(os.getcwd())
print("\nFiles in current folder:")
print(os.listdir())

d:\Solo Hackathons\RazorPay\notebooks

Files in current folder:
['.gitkeep', '01_data_exploration.ipynb']


In [102]:
import os

data_path = "../data"

print("Data folder:")
print(os.listdir(data_path))

Data folder:
['.gitkeep', 'external', 'processed', 'raw', 'README.md']


In [103]:
# Create recovery dataset from the current test set

recovery_data = test_data.copy()

# Model's predicted probability of payment failure
recovery_data["failure_probability"] = y_prob

# Expected revenue at risk
recovery_data["revenue_at_risk"] = (
    recovery_data["total_amount"]
    * recovery_data["failure_probability"]
)

# Rank transactions by revenue at risk
recovery_data = recovery_data.sort_values(
    "revenue_at_risk",
    ascending=False
).reset_index(drop=True)

print("Recovery dataset created successfully.")
print("Shape:", recovery_data.shape)

print("\nTop 10 transactions by revenue at risk:")

print(
    recovery_data[
        [
            "total_amount",
            "failure_probability",
            "revenue_at_risk",
            "payment_failed"
        ]
    ].head(10).to_string(index=False)
)

Recovery dataset created successfully.
Shape: (170517, 17)

Top 10 transactions by revenue at risk:
 total_amount  failure_probability  revenue_at_risk  payment_failed
     13249933             0.818853     1.084974e+07               0
     12301668             0.867448     1.067106e+07               0
     13828846             0.770681     1.065762e+07               1
     14587394             0.679455     9.911477e+06               0
     14876129             0.648219     9.642992e+06               0
     15270856             0.613130     9.363020e+06               0
     10499925             0.890416     9.349300e+06               0
      9012286             0.965738     8.703504e+06               0
      8525561             0.996378     8.494679e+06               0
      8444700             0.965691     8.154970e+06               1


In [104]:
total_revenue_at_risk = (
    recovery_data["revenue_at_risk"].sum()
)

actual_failed_revenue = (
    recovery_data.loc[
        recovery_data["payment_failed"] == 1,
        "total_amount"
    ].sum()
)

print("Total revenue at risk:", total_revenue_at_risk)
print("Actual failed revenue:", actual_failed_revenue)

Total revenue at risk: 40697706055.504005
Actual failed revenue: 4041413023


In [105]:
top_10_percent_count = int(
    len(recovery_data) * 0.10
)

top_risk = recovery_data.head(
    top_10_percent_count
)

actual_failed_in_top_risk = top_risk[
    top_risk["payment_failed"] == 1
]

print("Top 10% transactions:", len(top_risk))

print(
    "Actual failed transactions in top 10%:",
    len(actual_failed_in_top_risk)
)

print(
    "Failed revenue in top 10%:",
    actual_failed_in_top_risk["total_amount"].sum()
)

print(
    "Total transaction value in top 10%:",
    top_risk["total_amount"].sum()
)

Top 10% transactions: 17051
Actual failed transactions in top 10%: 1031
Failed revenue in top 10%: 1952604142
Total transaction value in top 10%: 39305179984


In [106]:
top_10_failed_revenue = (
    actual_failed_in_top_risk["total_amount"].sum()
)

capture_percentage = (
    top_10_failed_revenue
    / actual_failed_revenue
    * 100
)

print(
    "Failed-revenue captured by top 10%:",
    capture_percentage,
    "%"
)

Failed-revenue captured by top 10%: 48.31488716663147 %


In [107]:
print(
    recovery_data[
        [
            "created_at",
            "total_amount",
            "failure_probability",
            "revenue_at_risk",
            "payment_failed"
        ]
    ].head(25).to_string(index=False)
)

                      created_at  total_amount  failure_probability  revenue_at_risk  payment_failed
2022-05-07 00:26:03.603935+00:00      13249933             0.818853     1.084974e+07               0
2022-03-15 15:16:53.266157+00:00      12301668             0.867448     1.067106e+07               0
2022-06-26 03:41:28.288857+00:00      13828846             0.770681     1.065762e+07               1
2022-03-20 13:39:57.847709+00:00      14587394             0.679455     9.911477e+06               0
2022-03-06 01:09:37.995110+00:00      14876129             0.648219     9.642992e+06               0
2022-05-14 03:59:21.992468+00:00      15270856             0.613130     9.363020e+06               0
2022-05-04 17:36:20.693897+00:00      10499925             0.890416     9.349300e+06               0
2022-07-29 17:18:41.622489+00:00       9012286             0.965738     8.703504e+06               0
2022-03-23 21:28:38.595801+00:00       8525561             0.996378     8.494679e+06       

In [108]:
recovery_data["amount_score"] = (
    recovery_data["total_amount"]
    / recovery_data["total_amount"].max()
)

In [110]:
recovery_data["history_score"] = (
    recovery_data["previous_failure_rate"]
    .fillna(0)
)

In [111]:
# ==========================================
# RECOVERY PRIORITY SCORING
# ==========================================

# Start fresh
recovery_data = test_data.copy()

# 1. Model failure probability
recovery_data["failure_probability"] = y_prob

# 2. Revenue at risk
recovery_data["revenue_at_risk"] = (
    recovery_data["total_amount"]
    * recovery_data["failure_probability"]
)

# 3. Transaction amount score
recovery_data["amount_score"] = (
    recovery_data["total_amount"]
    / recovery_data["total_amount"].max()
)

# 4. Customer history score
recovery_data["history_score"] = (
    recovery_data["previous_failure_rate"]
    .fillna(0)
)

# 5. Recovery priority score
recovery_data["recovery_priority_score"] = (
    0.50 * recovery_data["failure_probability"]
    + 0.30 * recovery_data["amount_score"]
    + 0.20 * recovery_data["history_score"]
)

# 6. Recovery priority category
def assign_recovery_priority(score):

    if score >= 0.70:
        return "Critical"

    elif score >= 0.50:
        return "High"

    elif score >= 0.30:
        return "Medium"

    else:
        return "Low"


recovery_data["recovery_priority"] = (
    recovery_data["recovery_priority_score"]
    .apply(assign_recovery_priority)
)

# 7. Sort by priority score
recovery_data = recovery_data.sort_values(
    "recovery_priority_score",
    ascending=False
).reset_index(drop=True)


# ==========================================
# RESULTS
# ==========================================

print("Recovery scoring completed successfully.")

print("\nPriority distribution:")
print(
    recovery_data["recovery_priority"]
    .value_counts()
)

print("\nTop 20 recovery priorities:")

print(
    recovery_data[
        [
            "customer_id",
            "total_amount",
            "failure_probability",
            "previous_failures",
            "previous_failure_rate",
            "revenue_at_risk",
            "recovery_priority_score",
            "recovery_priority",
            "payment_failed"
        ]
    ].head(20).to_string(index=False)
)

Recovery scoring completed successfully.

Priority distribution:
recovery_priority
Low         140401
Medium       26438
High          3641
Critical        37
Name: count, dtype: int64

Top 20 recovery priorities:


KeyError: "['customer_id'] not in index"

In [112]:
print("Recovery data columns:")
print(recovery_data.columns.tolist())

Recovery data columns:
['created_at', 'total_amount', 'promo_amount', 'shipment_fee', 'previous_transactions', 'previous_failures', 'previous_successes', 'previous_failure_rate', 'previous_spending', 'previous_average_amount', 'payment_failed', 'payment_method_Debit Card', 'payment_method_Gopay', 'payment_method_LinkAja', 'payment_method_OVO', 'failure_probability', 'revenue_at_risk', 'amount_score', 'history_score', 'recovery_priority_score', 'recovery_priority']


In [113]:
required_columns = [
    "customer_id",
    "total_amount",
    "failure_probability",
    "previous_failures",
    "previous_failure_rate",
    "revenue_at_risk",
    "recovery_priority_score",
    "recovery_priority",
    "payment_failed"
]

print("\nMissing columns:")
print([
    col for col in required_columns
    if col not in recovery_data.columns
])


Missing columns:
['customer_id']


In [114]:
print(
    recovery_data[
        [
            "created_at",
            "total_amount",
            "failure_probability",
            "previous_failures",
            "previous_failure_rate",
            "revenue_at_risk",
            "recovery_priority_score",
            "recovery_priority",
            "payment_failed"
        ]
    ].head(20).to_string(index=False)
)

                      created_at  total_amount  failure_probability  previous_failures  previous_failure_rate  revenue_at_risk  recovery_priority_score recovery_priority  payment_failed
2022-07-29 17:18:41.622489+00:00       9012286             0.965738                1.0                    1.0     8.703504e+06                 0.807441          Critical               0
2022-06-16 16:14:19.559706+00:00       4686848             0.967410                1.0                    1.0     4.534104e+06                 0.748489          Critical               0
2022-02-09 16:33:56.684062+00:00       4614023             0.967902                1.0                    1.0     4.465923e+06                 0.747728          Critical               0
2022-03-30 13:39:16.921698+00:00       4479366             0.969362                1.0                    1.0     4.342127e+06                 0.746597          Critical               0
2022-07-10 05:39:00.609426+00:00       4856242             0.957968   

In [115]:
print(
    recovery_data["recovery_priority"]
    .value_counts()
)

recovery_priority
Low         140401
Medium       26438
High          3641
Critical        37
Name: count, dtype: int64


In [116]:
def assign_recovery_action(priority):

    if priority == "Critical":
        return "Immediate Recovery"

    elif priority == "High":
        return "Retry Payment"

    elif priority == "Medium":
        return "Alternative Payment Method"

    else:
        return "No Immediate Action"


recovery_data["recovery_action"] = (
    recovery_data["recovery_priority"]
    .apply(assign_recovery_action)
)

print(
    recovery_data["recovery_action"]
    .value_counts()
)

recovery_action
No Immediate Action           140401
Alternative Payment Method     26438
Retry Payment                   3641
Immediate Recovery                37
Name: count, dtype: int64


In [117]:
print(
    recovery_data[
        [
            "total_amount",
            "failure_probability",
            "revenue_at_risk",
            "recovery_priority_score",
            "recovery_priority",
            "recovery_action",
            "payment_failed"
        ]
    ].head(30).to_string(index=False)
)

 total_amount  failure_probability  revenue_at_risk  recovery_priority_score recovery_priority    recovery_action  payment_failed
      9012286             0.965738     8.703504e+06                 0.807441          Critical Immediate Recovery               0
      4686848             0.967410     4.534104e+06                 0.748489          Critical Immediate Recovery               0
      4614023             0.967902     4.465923e+06                 0.747728          Critical Immediate Recovery               0
      4479366             0.969362     4.342127e+06                 0.746597          Critical Immediate Recovery               0
      4856242             0.957968     4.652126e+06                 0.746109          Critical Immediate Recovery               0
      3077675             0.968139     2.979616e+06                 0.726610          Critical Immediate Recovery               0
      2780268             0.973438     2.706418e+06                 0.725149          Crit

In [118]:
priority_revenue = (
    recovery_data[
        recovery_data["payment_failed"] == 1
    ]
    .groupby("recovery_priority")["total_amount"]
    .agg(["count", "sum"])
    .sort_values("sum", ascending=False)
)

print(priority_revenue)

                   count         sum
recovery_priority                   
Low                 3831  1920888934
Medium              2891  1656170186
High                 697   455937522
Critical               6     8416381


In [119]:
recovery_opportunity = (
    recovery_data[
        recovery_data["payment_failed"] == 1
    ]
    .groupby("recovery_action")["total_amount"]
    .agg(["count", "sum"])
    .sort_values("sum", ascending=False)
)

print(recovery_opportunity)

                            count         sum
recovery_action                              
No Immediate Action          3831  1920888934
Alternative Payment Method   2891  1656170186
Retry Payment                 697   455937522
Immediate Recovery              6     8416381


In [120]:
payment_method_analysis = (
    df.groupby("payment_method")["payment_status"]
    .agg(
        total_transactions="count",
        failed_transactions=lambda x: (x == "Failed").sum()
    )
)

payment_method_analysis["failure_rate"] = (
    payment_method_analysis["failed_transactions"]
    / payment_method_analysis["total_transactions"]
)

payment_method_analysis = (
    payment_method_analysis
    .sort_values("failure_rate", ascending=False)
)

print(payment_method_analysis)

                total_transactions  failed_transactions  failure_rate
payment_method                                                       
LinkAja                      75329                 3323      0.044113
Debit Card                  137269                 5958      0.043404
Gopay                       171334                 7387      0.043115
Credit Card                 299586                12894      0.043039
OVO                         169066                 7058      0.041747


In [124]:
failed_revenue_by_method = (
    df[df["payment_status"] == "Failed"]
    .groupby("payment_method")["total_amount"]
    .agg(
        failed_transactions="count",
        failed_revenue="sum"
    )
    .sort_values("failed_revenue", ascending=False)
)



In [125]:
method_status = pd.crosstab(
    df["payment_method"],
    df["payment_status"]
)

print(method_status)

payment_status  Failed  Success
payment_method                 
Credit Card      12894   286692
Debit Card        5958   131311
Gopay             7387   163947
LinkAja           3323    72006
OVO               7058   162008


In [126]:
customer_analysis = (
    df.groupby("customer_id")["payment_status"]
    .agg(
        total_transactions="count",
        failed_transactions=lambda x: (x == "Failed").sum()
    )
)

customer_analysis["failure_rate"] = (
    customer_analysis["failed_transactions"]
    / customer_analysis["total_transactions"]
)

print(customer_analysis.describe())

       total_transactions  failed_transactions  failure_rate
count        50705.000000         50705.000000  50705.000000
mean            16.814594             0.722217      0.043341
std             29.711045             2.311395      0.126118
min              1.000000             0.000000      0.000000
25%              2.000000             0.000000      0.000000
50%              6.000000             0.000000      0.000000
75%             18.000000             1.000000      0.009009
max            550.000000            83.000000      1.000000


In [127]:
top_failed_customers = (
    customer_analysis
    .sort_values(
        "failed_transactions",
        ascending=False
    )
    .head(20)
)

print(top_failed_customers)

             total_transactions  failed_transactions  failure_rate
customer_id                                                       
21093                       283                   83      0.293286
62854                       205                   74      0.360976
33230                       146                   66      0.452055
76412                       197                   57      0.289340
34185                       175                   50      0.285714
91425                       187                   50      0.267380
31173                       267                   50      0.187266
30956                       207                   47      0.227053
17434                       320                   46      0.143750
95492                       401                   44      0.109726
81701                       164                   43      0.262195
99070                       154                   42      0.272727
21638                       157                   42      0.26

In [128]:
high_risk_customers = (
    customer_analysis[
        customer_analysis["total_transactions"] >= 10
    ]
    .sort_values(
        "failure_rate",
        ascending=False
    )
    .head(20)
)

print(high_risk_customers)

             total_transactions  failed_transactions  failure_rate
customer_id                                                       
22283                        15                    9      0.600000
45109                        10                    6      0.600000
20418                        12                    7      0.583333
2850                         12                    7      0.583333
78154                        16                    9      0.562500
24645                        36                   20      0.555556
80648                        11                    6      0.545455
77975                        13                    7      0.538462
74166                        15                    8      0.533333
78729                        12                    6      0.500000
11139                        14                    7      0.500000
33895                        14                    7      0.500000
57259                        10                    5      0.50

In [129]:
failure_history_comparison = recovery_data.copy()

failure_history_comparison["has_previous_failures"] = (
    failure_history_comparison["previous_failures"] > 0
)

print(
    failure_history_comparison.groupby(
        "has_previous_failures"
    )["payment_failed"]
    .agg(
        transactions="count",
        actual_failures="sum"
    )
)

                       transactions  actual_failures
has_previous_failures                               
False                         98131             2058
True                          72386             5367


In [130]:
history_failure_rate = (
    failure_history_comparison
    .groupby("has_previous_failures")["payment_failed"]
    .mean()
)

print(history_failure_rate)

has_previous_failures
False    0.020972
True     0.074144
Name: payment_failed, dtype: float64


In [131]:
# Customer history risk score

recovery_data["history_score"] = (
    recovery_data["previous_failure_rate"].fillna(0)
)

# Reduce the effect of extreme rates for customers
# with very little transaction history
recovery_data["history_confidence"] = (
    recovery_data["previous_transactions"]
    / (recovery_data["previous_transactions"] + 10)
)

recovery_data["adjusted_history_score"] = (
    recovery_data["history_score"]
    * recovery_data["history_confidence"]
)

print(
    recovery_data[
        [
            "previous_transactions",
            "previous_failures",
            "previous_failure_rate",
            "history_confidence",
            "adjusted_history_score"
        ]
    ].head(20).to_string(index=False)
)

 previous_transactions  previous_failures  previous_failure_rate  history_confidence  adjusted_history_score
                     1                1.0                    1.0            0.090909                0.090909
                     1                1.0                    1.0            0.090909                0.090909
                     1                1.0                    1.0            0.090909                0.090909
                     1                1.0                    1.0            0.090909                0.090909
                     1                1.0                    1.0            0.090909                0.090909
                     1                1.0                    1.0            0.090909                0.090909
                     2                2.0                    1.0            0.166667                0.166667
                     1                1.0                    1.0            0.090909                0.090909
                   

In [132]:
recovery_data["recovery_priority_score"] = (
    0.50 * recovery_data["failure_probability"]
    + 0.30 * recovery_data["amount_score"]
    + 0.20 * recovery_data["adjusted_history_score"]
)

recovery_data["recovery_priority"] = (
    recovery_data["recovery_priority_score"]
    .apply(assign_recovery_priority)
)

recovery_data = recovery_data.sort_values(
    "recovery_priority_score",
    ascending=False
).reset_index(drop=True)

print(
    recovery_data["recovery_priority"]
    .value_counts()
)

recovery_priority
Low       141987
Medium     26033
High        2497
Name: count, dtype: int64


In [133]:
print(
    recovery_data[
        [
            "created_at",
            "total_amount",
            "failure_probability",
            "previous_transactions",
            "previous_failures",
            "previous_failure_rate",
            "adjusted_history_score",
            "revenue_at_risk",
            "recovery_priority_score",
            "recovery_priority",
            "payment_failed"
        ]
    ].head(20).to_string(index=False)
)

                      created_at  total_amount  failure_probability  previous_transactions  previous_failures  previous_failure_rate  adjusted_history_score  revenue_at_risk  recovery_priority_score recovery_priority  payment_failed
2022-03-23 21:28:38.595801+00:00       8525561             0.996378                    128               37.0               0.289062                0.268116     8.494679e+06                 0.669656              High               0
2022-02-27 19:58:10.743040+00:00       6181658             0.999986                    189               70.0               0.370370                0.351759     6.181570e+06                 0.655790              High               1
2022-05-04 17:36:20.693897+00:00      10499925             0.890416                     21                8.0               0.380952                0.258065     9.349300e+06                 0.641956              High               0
2022-06-13 03:13:00.096510+00:00       8444700             0.965691 

In [134]:
def assign_recovery_priority(score):

    if score >= 0.60:
        return "Critical"

    elif score >= 0.50:
        return "High"

    elif score >= 0.30:
        return "Medium"

    else:
        return "Low"


recovery_data["recovery_priority"] = (
    recovery_data["recovery_priority_score"]
    .apply(assign_recovery_priority)
)

recovery_data = recovery_data.sort_values(
    "recovery_priority_score",
    ascending=False
).reset_index(drop=True)

print(
    recovery_data["recovery_priority"]
    .value_counts()
)

recovery_priority
Low         141987
Medium       26033
High          2476
Critical        21
Name: count, dtype: int64


In [135]:
recovery_data["recovery_action"] = (
    recovery_data["recovery_priority"]
    .apply(assign_recovery_action)
)

print(
    recovery_data["recovery_action"]
    .value_counts()
)

recovery_action
No Immediate Action           141987
Alternative Payment Method     26033
Retry Payment                   2476
Immediate Recovery                21
Name: count, dtype: int64


In [136]:
recovery_opportunity = (
    recovery_data[
        recovery_data["payment_failed"] == 1
    ]
    .groupby("recovery_action")["total_amount"]
    .agg(
        failed_transactions="count",
        failed_revenue="sum"
    )
    .sort_values(
        "failed_revenue",
        ascending=False
    )
)

print(recovery_opportunity)

                            failed_transactions  failed_revenue
recovery_action                                                
No Immediate Action                        3949      1990814847
Alternative Payment Method                 2957      1707182962
Retry Payment                               515       309683107
Immediate Recovery                            4        33732107


In [137]:
recovery_rates = {
    "Immediate Recovery": 0.70,
    "Retry Payment": 0.50,
    "Alternative Payment Method": 0.40,
    "No Immediate Action": 0.00
}

recovery_data["estimated_recovery_rate"] = (
    recovery_data["recovery_action"]
    .map(recovery_rates)
)

recovery_data["estimated_recoverable_revenue"] = (
    recovery_data["revenue_at_risk"]
    * recovery_data["estimated_recovery_rate"]
)

print(
    recovery_data[
        [
            "total_amount",
            "failure_probability",
            "revenue_at_risk",
            "recovery_priority",
            "recovery_action",
            "estimated_recovery_rate",
            "estimated_recoverable_revenue"
        ]
    ].head(20).to_string(index=False)
)

 total_amount  failure_probability  revenue_at_risk recovery_priority    recovery_action  estimated_recovery_rate  estimated_recoverable_revenue
      8525561             0.996378     8.494679e+06          Critical Immediate Recovery                      0.7                   5.946275e+06
      6181658             0.999986     6.181570e+06          Critical Immediate Recovery                      0.7                   4.327099e+06
     10499925             0.890416     9.349300e+06          Critical Immediate Recovery                      0.7                   6.544510e+06
      8444700             0.965691     8.154970e+06          Critical Immediate Recovery                      0.7                   5.708479e+06
     13249933             0.818853     1.084974e+07          Critical Immediate Recovery                      0.7                   7.594819e+06
      5409263             0.999994     5.409233e+06          Critical Immediate Recovery                      0.7                 

In [138]:
total_at_risk = recovery_data["revenue_at_risk"].sum()

estimated_recoverable = (
    recovery_data["estimated_recoverable_revenue"].sum()
)

actual_failed_revenue = (
    recovery_data[
        recovery_data["payment_failed"] == 1
    ]["total_amount"].sum()
)

print("Total revenue at risk:", total_at_risk)
print("Actual failed revenue:", actual_failed_revenue)
print("Estimated recoverable revenue:", estimated_recoverable)

print(
    "Estimated recovery opportunity:",
    estimated_recoverable / actual_failed_revenue * 100,
    "%"
)

Total revenue at risk: 40697706055.50401
Actual failed revenue: 4041413023
Estimated recoverable revenue: 5278652259.400988
Estimated recovery opportunity: 130.61402606859932 %


In [139]:
# ==========================================
# FINAL RECOVERY BUSINESS METRICS
# ==========================================

# Actual failed revenue
actual_failed_revenue = recovery_data.loc[
    recovery_data["payment_failed"] == 1,
    "total_amount"
].sum()

# Revenue at risk identified by the model
total_revenue_at_risk = recovery_data[
    "revenue_at_risk"
].sum()

# Estimated recovery value
estimated_recoverable_revenue = recovery_data[
    "estimated_recoverable_revenue"
].sum()

# Estimated recovery value as percentage of modelled revenue at risk
estimated_recovery_of_risk = (
    estimated_recoverable_revenue
    / total_revenue_at_risk
    * 100
)

print("====================================")
print("REVENUE RECOVERY METRICS")
print("====================================")

print(
    f"Actual failed revenue: "
    f"{actual_failed_revenue:,.2f}"
)

print(
    f"Modelled revenue at risk: "
    f"{total_revenue_at_risk:,.2f}"
)

print(
    f"Estimated recoverable value: "
    f"{estimated_recoverable_revenue:,.2f}"
)

print(
    f"Estimated recovery of modelled risk: "
    f"{estimated_recovery_of_risk:.2f}%"
)

REVENUE RECOVERY METRICS
Actual failed revenue: 4,041,413,023.00
Modelled revenue at risk: 40,697,706,055.50
Estimated recoverable value: 5,278,652,259.40
Estimated recovery of modelled risk: 12.97%


In [140]:
priority_summary = (
    recovery_data
    .groupby("recovery_priority")
    .agg(
        transactions=("total_amount", "count"),
        total_value=("total_amount", "sum"),
        revenue_at_risk=("revenue_at_risk", "sum"),
        estimated_recoverable=(
            "estimated_recoverable_revenue",
            "sum"
        )
    )
    .sort_values(
        "estimated_recoverable",
        ascending=False
    )
)

print(priority_summary)

                   transactions  total_value  revenue_at_risk  \
recovery_priority                                               
Medium                    26033  17044559778     1.090295e+10   
High                       2476   1795703351     1.648381e+09   
Critical                     21    143622927     1.332601e+08   
Low                      141987  74748539249     2.801312e+10   

                   estimated_recoverable  
recovery_priority                         
Medium                      4.361180e+09  
High                        8.241906e+08  
Critical                    9.328210e+07  
Low                         0.000000e+00  


In [141]:
print("\nEstimated recoverable value by priority:")
print(
    priority_summary[
        ["revenue_at_risk", "estimated_recoverable"]
    ]
)


Estimated recoverable value by priority:
                   revenue_at_risk  estimated_recoverable
recovery_priority                                        
Medium                1.090295e+10           4.361180e+09
High                  1.648381e+09           8.241906e+08
Critical              1.332601e+08           9.328210e+07
Low                   2.801312e+10           0.000000e+00


In [142]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "recovery_scored_transactions.csv"

recovery_data.to_csv(
    output_file,
    index=False
)

print("Saved successfully!")
print("File:", output_file)
print("Rows:", len(recovery_data))
print("Columns:", len(recovery_data.columns))

Saved successfully!
File: ..\data\processed\recovery_scored_transactions.csv
Rows: 170517
Columns: 26


In [143]:
check_data = pd.read_csv(output_file)

print("Shape:", check_data.shape)
print("\nColumns:")
print(check_data.columns.tolist())

print("\nFirst 5 rows:")
print(check_data.head().to_string(index=False))

Shape: (170517, 26)

Columns:
['created_at', 'total_amount', 'promo_amount', 'shipment_fee', 'previous_transactions', 'previous_failures', 'previous_successes', 'previous_failure_rate', 'previous_spending', 'previous_average_amount', 'payment_failed', 'payment_method_Debit Card', 'payment_method_Gopay', 'payment_method_LinkAja', 'payment_method_OVO', 'failure_probability', 'revenue_at_risk', 'amount_score', 'history_score', 'recovery_priority_score', 'recovery_priority', 'recovery_action', 'history_confidence', 'adjusted_history_score', 'estimated_recovery_rate', 'estimated_recoverable_revenue']

First 5 rows:
                      created_at  total_amount  promo_amount  shipment_fee  previous_transactions  previous_failures  previous_successes  previous_failure_rate  previous_spending  previous_average_amount  payment_failed  payment_method_Debit Card  payment_method_Gopay  payment_method_LinkAja  payment_method_OVO  failure_probability  revenue_at_risk  amount_score  history_score  r